# AutoML Experiments

This notebook evaluates FLAML and AutoGluon as independent binary AutoML searches for every competency in the authoritative 53-label universe. It executes the shared five-fold cross-validation and fixed-test training-size protocols with resumable, label-level checkpoints.


In [ ]:
from pathlib import Path

EXPERIMENT_OUTPUT_ROOT = Path("results/experiments")
EXPERIMENT_SEED = 42
PROTOCOL_ASSIGNMENT_SEED = 42
PROTOCOL_VERSION = "protocol-v1"
RESUME_COMPLETED_UNITS = True
FORCE_RERUN = False
EXPECTED_LABEL_COUNT = 53
CV_N_SPLITS = 5
TRAIN_SIZE_REPETITIONS = 3
TRAIN_SIZES = [270, 135, 68]
AUTOML_NOMINAL_BUDGET_SECONDS = 600
AUTOML_PER_LABEL_BUDGET_SECONDS = AUTOML_NOMINAL_BUDGET_SECONDS / EXPECTED_LABEL_COUNT
AUTOML_BUDGET_PROTOCOL = "automl_equal_budget"
AUTOML_METHOD_MATRIX = [
    ("flaml", "tfidf"),
    ("flaml", "word2vec"),
    ("flaml", "bert"),
    ("autogluon", "tfidf"),
    ("autogluon", "word2vec"),
    ("autogluon", "bert"),
]


## Shared protocol, partition-local features, and label checkpoints

TF-IDF and Word2Vec are fitted only on the current unit's training courses. BERT is a fixed feature extractor cached with ordered course IDs and then indexed for each unit.


In [ ]:
import hashlib
import importlib.metadata
import json
import os
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from pipeline_utils import *

AUTOML_BUDGET_ALLOCATION = allocate_nominal_budget(AUTOML_NOMINAL_BUDGET_SECONDS, EXPECTED_LABEL_COUNT)


def read_selected_artifact(selected_attempt, artifact_name, *, kind):
    manifest = validate_completed_attempt(
        selected_attempt, expected_unit_key=selected_attempt.parent.name
    )
    if manifest is None or artifact_name not in manifest["artifacts"]:
        raise ValueError(f"Selected artifact is missing or invalid: {artifact_name}")
    artifact_path = selected_attempt / manifest["artifacts"][artifact_name]["path"]
    if kind == "csv":
        return pd.read_csv(artifact_path)
    if kind == "json":
        return json.loads(artifact_path.read_text(encoding="utf-8"))
    raise ValueError(f"Unsupported selected artifact kind: {kind}")


def build_run_manifest(unit_key, unit_spec, provenance):
    return {
        "unit_key": unit_key,
        "unit_spec": unit_spec,
        "provenance": provenance,
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
    }


def validate_run_manifest(manifest):
    required = {"unit_key", "unit_spec", "provenance", "started_at_utc"}
    if set(manifest) != required:
        raise ValueError("Run manifest schema is invalid")
    if experiment_unit_key(manifest["unit_spec"]) != manifest["unit_key"]:
        raise ValueError("Run manifest identity is invalid")
    return manifest


def validate_label_partition_manifest(y_train):
    active_label_indices, constant_labels = partition_label_columns(y_train)
    if active_label_indices != sorted(active_label_indices):
        raise ValueError("Active label indices must be ordered")
    expected = set(range(y_train.shape[1]))
    if set(active_label_indices) | set(constant_labels) != expected:
        raise ValueError("Active and constant labels must form a complete partition")
    if set(active_label_indices) & set(constant_labels):
        raise ValueError("Active and constant label sets must be disjoint")
    return active_label_indices, constant_labels


def build_unit_manifest(unit_spec, active_label_indices, constant_labels):
    return {
        "unit_spec": unit_spec,
        "active_label_indices": list(active_label_indices),
        "constant_labels": {
            str(index): int(value) for index, value in sorted(constant_labels.items())
        },
    }


def report_progress_counters(progress_counters):
    progress_counters["remaining"] = max(
        0,
        progress_counters["total"]
        - progress_counters["completed"]
        - progress_counters["skipped"]
        - progress_counters["failed"],
    )
    print(
        "AutoML progress:",
        ", ".join(f"{key}={value}" for key, value in progress_counters.items()),
    )


def _tfidf_features(train_texts, eval_texts):
    from sklearn.feature_extraction.text import TfidfVectorizer

    vectorizer = TfidfVectorizer(
        max_features=2000,
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.7,
        sublinear_tf=True,
    )
    train_features = vectorizer.fit_transform(train_texts).toarray()
    eval_features = vectorizer.transform(eval_texts).toarray()
    identity = {
        "kind": "tfidf",
        "preprocessing": {
            "max_features": 2000,
            "ngram_range": [1, 2],
            "min_df": 5,
            "max_df": 0.7,
            "sublinear_tf": True,
            "fit_scope": "unit_train_only",
        },
    }
    return train_features, eval_features, identity


def _word2vec_features(train_texts, eval_texts, *, model_seed):
    import re
    from gensim.models import Word2Vec

    def tokenize(text):
        return re.findall(r"[^\W\d_]+", str(text).lower(), flags=re.UNICODE)

    train_tokens = [tokenize(text) for text in train_texts]
    eval_tokens = [tokenize(text) for text in eval_texts]
    model = Word2Vec(
        sentences=train_tokens,
        vector_size=300,
        window=5,
        min_count=2,
        workers=1,
        seed=model_seed,
    )

    def embed(rows):
        embedded = []
        for tokens in rows:
            vectors = [model.wv[token] for token in tokens if token in model.wv]
            embedded.append(np.mean(vectors, axis=0) if vectors else np.zeros(300))
        return np.vstack(embedded)

    identity = {
        "kind": "word2vec",
        "preprocessing": {
            "vector_size": 300,
            "window": 5,
            "min_count": 2,
            "workers": 1,
            "seed": model_seed,
            "fit_scope": "unit_train_only",
        },
    }
    return embed(train_tokens), embed(eval_tokens), identity


def _get_bert_embeddings(texts):
    import torch
    from transformers import AutoModel, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_ID)
    model = AutoModel.from_pretrained(BERT_MODEL_ID)
    model.eval()
    embeddings = []
    for start in range(0, len(texts), BERT_BATCH_SIZE):
        batch = texts[start : start + BERT_BATCH_SIZE]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=BERT_MAX_LENGTH,
        )
        with torch.no_grad():
            output = model(**inputs).last_hidden_state[:, 0, :]
        embeddings.append(output.cpu().numpy())
    return np.vstack(embeddings), {
        "resolved_model_revision": getattr(model.config, "_commit_hash", None) or "unresolved-local",
        "resolved_tokenizer_revision": getattr(tokenizer, "_commit_hash", None) or "unresolved-local",
    }


def automl_bert_cache_metadata_matches(metadata, expected_identity):
    required_fields = {
        "source_path", "source_sha256", "ordered_course_ids_sha256", "model_id",
        "max_length", "batch_size", "expected_shape", "cache_key",
        "requested_model", "requested_tokenizer", "resolved_model_revision",
        "resolved_tokenizer_revision", "embedding_sha256",
    }
    if not isinstance(metadata, dict) or not required_fields <= set(metadata):
        return False
    request_fields = {
        "source_path", "source_sha256", "ordered_course_ids_sha256", "model_id",
        "max_length", "batch_size", "expected_shape", "requested_model",
        "requested_tokenizer",
    }
    if any(metadata[field] != expected_identity[field] for field in request_fields):
        return False
    if not metadata["resolved_model_revision"] or not metadata["resolved_tokenizer_revision"]:
        return False
    identity_payload = {
        field: metadata[field] for field in required_fields if field != "cache_key"
    }
    return metadata["cache_key"] == canonical_payload_hash(identity_payload)


def publish_file_exclusively(staged_path, destination_path):
    os.link(staged_path, destination_path)
    staged_path.unlink()


def validate_existing_bert_cache(cache_path, metadata_path, fresh_identity):
    if not cache_path.exists() or not metadata_path.exists():
        return None
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    if {key: metadata.get(key) for key in fresh_identity} != fresh_identity:
        return None
    payload = np.load(cache_path)
    embeddings = payload["embeddings"]
    cached_ids = payload["course_ids"].astype(str).tolist()
    observed_hash = hashlib.sha256(np.ascontiguousarray(embeddings).tobytes()).hexdigest()
    if observed_hash != fresh_identity["embedding_sha256"]:
        return None
    if embeddings.shape != tuple(fresh_identity["expected_shape"]):
        return None
    if cached_ids != course_ids.astype(str).tolist():
        return None
    return embeddings, metadata


def rebuild_automl_bert_cache_immutably(
    cache_dir,
    stale_cache_path,
    stale_metadata_path,
    expected_identity,
    fresh_embeddings,
    resolved_model_revision,
    resolved_tokenizer_revision,
):
    _ = (stale_cache_path, stale_metadata_path)
    if fresh_embeddings.shape != tuple(expected_identity["expected_shape"]):
        raise ValueError(f"Generated BERT shape mismatch: {fresh_embeddings.shape}")
    if not resolved_model_revision or not resolved_tokenizer_revision:
        raise ValueError("Resolved BERT model and tokenizer revisions are required")
    fresh_payload = {
        **expected_identity,
        "resolved_model_revision": resolved_model_revision,
        "resolved_tokenizer_revision": resolved_tokenizer_revision,
        "embedding_sha256": hashlib.sha256(
            np.ascontiguousarray(fresh_embeddings).tobytes()
        ).hexdigest(),
    }
    fresh_identity = {
        **fresh_payload,
        "cache_key": canonical_payload_hash(fresh_payload),
    }
    fresh_cache_path = cache_dir / f"bert_{fresh_identity['cache_key']}.npz"
    fresh_metadata_path = fresh_cache_path.with_suffix(".json")
    cache_pointer_path = cache_dir / "current_bert_cache.json"
    fresh_metadata = {
        **fresh_identity,
        "index_column": "courseId",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    if fresh_cache_path.exists() or fresh_metadata_path.exists():
        existing = validate_existing_bert_cache(
            fresh_cache_path, fresh_metadata_path, fresh_identity
        )
        if existing is not None:
            existing_embeddings, existing_metadata = existing
            atomic_write_json(cache_pointer_path, {
                "cache_key": fresh_identity["cache_key"],
                "cache_path": str(fresh_cache_path),
                "metadata_path": str(fresh_metadata_path),
            })
            return existing_embeddings, existing_metadata
        if fresh_cache_path.exists() and not fresh_metadata_path.exists():
            payload = np.load(fresh_cache_path)
            recovered_embeddings = payload["embeddings"]
            recovered_ids = payload["course_ids"].astype(str).tolist()
            recovered_hash = hashlib.sha256(
                np.ascontiguousarray(recovered_embeddings).tobytes()
            ).hexdigest()
            if (
                recovered_hash == fresh_identity["embedding_sha256"]
                and recovered_ids == course_ids.astype(str).tolist()
            ):
                staged_metadata = cache_dir / f".{fresh_identity['cache_key']}.metadata.tmp"
                atomic_write_json(staged_metadata, fresh_metadata)
                publish_file_exclusively(staged_metadata, fresh_metadata_path)
                atomic_write_json(cache_pointer_path, {
                    "cache_key": fresh_identity["cache_key"],
                    "cache_path": str(fresh_cache_path),
                    "metadata_path": str(fresh_metadata_path),
                })
                return recovered_embeddings, fresh_metadata
        raise ValueError("Content-addressed AutoML BERT cache key collision")

    staged_npz = cache_dir / f".{fresh_identity['cache_key']}.{os.getpid()}.tmp.npz"
    np.savez_compressed(
        staged_npz,
        embeddings=fresh_embeddings,
        course_ids=course_ids.astype(str),
    )
    publish_file_exclusively(staged_npz, fresh_cache_path)
    staged_metadata = cache_dir / f".{fresh_identity['cache_key']}.metadata.tmp"
    atomic_write_json(staged_metadata, fresh_metadata)
    publish_file_exclusively(staged_metadata, fresh_metadata_path)
    atomic_write_json(cache_pointer_path, {
        "cache_key": fresh_identity["cache_key"],
        "cache_path": str(fresh_cache_path),
        "metadata_path": str(fresh_metadata_path),
    })
    return fresh_embeddings, fresh_metadata


def load_or_build_indexed_bert_cache(all_course_ids, all_texts):
    cache_dir = EXPERIMENT_OUTPUT_ROOT / "cache" / "automl"
    cache_dir.mkdir(parents=True, exist_ok=True)
    base_identity = cache_identity(
        "processed.csv",
        all_course_ids,
        BERT_MODEL_ID,
        BERT_MAX_LENGTH,
        BERT_BATCH_SIZE,
        (len(all_course_ids), BERT_HIDDEN_SIZE),
    )
    expected_identity = {
        key: value for key, value in base_identity.items() if key != "cache_key"
    }
    expected_identity.update({
        "requested_model": BERT_MODEL_ID,
        "requested_tokenizer": BERT_MODEL_ID,
    })
    cache_pointer_path = cache_dir / "current_bert_cache.json"
    stale_cache_path = cache_dir / f"bert_{base_identity['cache_key']}.npz"
    stale_metadata_path = stale_cache_path.with_suffix(".json")
    if cache_pointer_path.exists():
        try:
            pointer = json.loads(cache_pointer_path.read_text(encoding="utf-8"))
            pointer_cache_path = Path(pointer["cache_path"])
            pointer_metadata_path = Path(pointer["metadata_path"])
            if (
                pointer_cache_path.parent.resolve() == cache_dir.resolve()
                and pointer_metadata_path.parent.resolve() == cache_dir.resolve()
            ):
                stale_cache_path = pointer_cache_path
                stale_metadata_path = pointer_metadata_path
        except (OSError, ValueError, KeyError, json.JSONDecodeError):
            pass
    if stale_cache_path.exists() and stale_metadata_path.exists():
        try:
            candidate_metadata = json.loads(stale_metadata_path.read_text(encoding="utf-8"))
            candidate_payload = np.load(stale_cache_path)
            candidate_embeddings = candidate_payload["embeddings"]
            candidate_ids = candidate_payload["course_ids"].astype(str).tolist()
        except (OSError, ValueError, KeyError, json.JSONDecodeError):
            candidate_metadata = {}
            candidate_embeddings = np.empty((0, 0))
            candidate_ids = []
        if not automl_bert_cache_metadata_matches(candidate_metadata, expected_identity):
            fresh_embeddings, resolved = _get_bert_embeddings(list(all_texts))
            return rebuild_automl_bert_cache_immutably(
                cache_dir,
                stale_cache_path,
                stale_metadata_path,
                expected_identity,
                fresh_embeddings,
                resolved["resolved_model_revision"],
                resolved["resolved_tokenizer_revision"],
            )
        observed_hash = hashlib.sha256(
            np.ascontiguousarray(candidate_embeddings).tobytes()
        ).hexdigest()
        if (
            candidate_ids == [str(value) for value in all_course_ids]
            and candidate_embeddings.shape == tuple(expected_identity["expected_shape"])
            and candidate_metadata["embedding_sha256"] == observed_hash
        ):
            return candidate_embeddings, candidate_metadata
    fresh_embeddings, resolved = _get_bert_embeddings(list(all_texts))
    return rebuild_automl_bert_cache_immutably(
        cache_dir,
        stale_cache_path,
        stale_metadata_path,
        expected_identity,
        fresh_embeddings,
        resolved["resolved_model_revision"],
        resolved["resolved_tokenizer_revision"],
    )
def prepare_unit_features(representation, train_indices, eval_indices, *, model_seed):
    train_texts = processed_data.iloc[train_indices]["combinedText"].astype(str).tolist()
    eval_texts = processed_data.iloc[eval_indices]["combinedText"].astype(str).tolist()
    if representation == "tfidf":
        return _tfidf_features(train_texts, eval_texts)
    if representation == "word2vec":
        return _word2vec_features(train_texts, eval_texts, model_seed=model_seed)
    if representation == "bert":
        identity = {
            "kind": "bert",
            **bert_identity,
            "embedding_cache_identity": embedding_cache_identity,
        }
        return bert_embeddings[train_indices], bert_embeddings[eval_indices], identity
    raise ValueError(f"Unsupported AutoML representation: {representation}")


def build_feature_identity(representation, model_seed):
    if representation == "tfidf":
        return {
            "kind": "tfidf",
            "preprocessing": {
                "max_features": 2000,
                "ngram_range": [1, 2],
                "min_df": 5,
                "max_df": 0.7,
                "sublinear_tf": True,
                "fit_scope": "unit_train_only",
            },
        }
    if representation == "word2vec":
        return {
            "kind": "word2vec",
            "preprocessing": {
                "vector_size": 300,
                "window": 5,
                "min_count": 2,
                "workers": 1,
                "seed": model_seed,
                "fit_scope": "unit_train_only",
            },
        }
    if representation == "bert":
        return {
            "kind": "bert",
            **bert_identity,
            "embedding_cache_identity": embedding_cache_identity,
        }
    raise ValueError(f"Unsupported AutoML representation: {representation}")


def build_automl_model_config(framework, model_seed):
    common = {
        "framework": framework,
        "seed": model_seed,
        "training_order_seed": model_seed,
        "n_labels": 53,
        "nominal_budget_seconds": 600,
        "per_label_budget_seconds": 600 / 53,
        "budget_allocation_formula": "600 / 53",
        "budget_scope": "per_representation",
        "no_redistribution": True,
        "overhead_excluded": True,
        "framework_overhead_note": (
            "The nominal per-label search budget excludes framework startup, "
            "serialization, checkpoint validation, and inference overhead."
        ),
    }
    if framework == "flaml":
        return {
            **common,
            "task": "classification",
            "metric": "f1",
            "time_budget": 600 / 53,
        }
    if framework == "autogluon":
        return {
            **common,
            "problem_type": "binary",
            "presets": "medium_quality_faster_train",
            "eval_metric": "f1",
            "time_limit": 600 / 53,
        }
    raise ValueError(f"Unsupported AutoML framework: {framework}")


def _label_configuration(model_config):
    return dict(model_config)


def _load_flaml_checkpoint(path):
    import joblib

    path = Path(path)
    if path.is_dir():
        path = path / "estimator.joblib"
    return joblib.load(path)


def _load_autogluon_checkpoint(path):
    from autogluon.tabular import TabularPredictor

    path = Path(path)
    if path.is_dir() and (path / "completed.json").is_file():
        marker = json.loads((path / "completed.json").read_text(encoding="utf-8"))
        path = path / marker["artifacts"]["model"]["path"]
    return TabularPredictor.load(str(path))


def _load_constant_checkpoint(path):
    path = Path(path)
    if path.is_dir():
        path = path / "constant.json"
    return json.loads(path.read_text(encoding="utf-8"))


def _marker_artifact(checkpoint_dir, default_relative):
    marker_path = checkpoint_dir / "completed.json"
    if not marker_path.exists():
        return default_relative
    try:
        marker = json.loads(marker_path.read_text(encoding="utf-8"))
        candidate = marker["artifacts"]["model"]["path"]
    except (OSError, KeyError, TypeError, json.JSONDecodeError):
        return default_relative
    candidate_path = Path(candidate)
    if candidate_path.is_absolute() or ".." in candidate_path.parts:
        return default_relative
    return candidate


def _fit_flaml_label(X_train, y_train, *, budget_seconds, metric, model_seed, model_path):
    import joblib
    from flaml import AutoML

    search = AutoML()
    search.fit(
        X_train=X_train,
        y_train=y_train,
        time_budget=budget_seconds,
        metric=metric,
        task="classification",
        seed=model_seed,
    )
    estimator = search.model.estimator
    joblib.dump(estimator, model_path)
    return estimator


def _fit_autogluon_label(X_train, y_train, *, budget_seconds, metric, model_seed, model_path):
    import random
    from autogluon.tabular import TabularPredictor

    random.seed(model_seed)
    np.random.seed(model_seed)
    feature_names = [f"f_{index}" for index in range(X_train.shape[1])]
    train_frame = pd.DataFrame(X_train, columns=feature_names)
    train_frame["target"] = y_train
    training_order = np.random.permutation(len(train_frame))
    ordered_train_frame = train_frame.iloc[training_order].reset_index(drop=True)
    predictor = TabularPredictor(
        label="target",
        path=str(model_path),
        problem_type="binary",
        eval_metric=metric,
    )
    predictor.fit(
        train_data=ordered_train_frame,
        time_limit=budget_seconds,
        presets="medium_quality_faster_train",
    )
    return predictor
def _binary_positive_probability(model, framework, X_eval, observed_classes):
    if framework == "flaml":
        probabilities = np.asarray(model.predict_proba(X_eval), dtype=float)
        classes = list(getattr(model, "classes_", observed_classes))
    elif framework == "autogluon":
        frame = pd.DataFrame(
            X_eval, columns=[f"f_{index}" for index in range(X_eval.shape[1])]
        )
        predicted = model.predict_proba(frame, as_multiclass=True)
        probabilities = np.asarray(predicted, dtype=float)
        classes = list(predicted.columns)
    else:
        raise ValueError(f"Unsupported AutoML framework: {framework}")
    if 1 not in classes:
        return np.full(len(X_eval), float(classes[0] == 1))
    return probabilities[:, classes.index(1)]


def load_label_fit_timing(fit_timing_path):
    payload = json.loads(Path(fit_timing_path).read_text(encoding="utf-8"))
    fit_seconds = float(payload["fit_seconds"])
    if not np.isfinite(fit_seconds) or fit_seconds < 0:
        raise ValueError("Label fit timing must be finite and non-negative")
    return fit_seconds


def run_automl_label_searches(
    framework,
    X_train,
    y_train,
    authoritative_labels,
    attempt,
    model_config,
    *,
    model_seed,
):
    active_label_indices, constant_labels = validate_label_partition_manifest(y_train)
    fitted_label_models = {}
    label_fit_seconds = []
    checkpoint_load_seconds = 0.0
    active_index_set = set(active_label_indices)
    for label_index, label_name in enumerate(authoritative_labels):
        observed_classes = sorted(np.unique(y_train[:, label_index]).astype(int).tolist())
        checkpoint_configuration = _label_configuration(model_config)
        checkpoint_dir = attempt.path / "labels" / f"label-{label_index:03d}"
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        fit_timing_path = checkpoint_dir / "fit_timing.json"
        if label_index not in active_index_set:
            constant_path = checkpoint_dir / "constant.json"
            checkpoint_started = time.perf_counter()
            valid = validate_label_checkpoint(
                checkpoint_dir,
                unit_key=attempt.unit_key,
                label_index=label_index,
                label_name=label_name,
                configuration=checkpoint_configuration,
                observed_classes=observed_classes,
                expected_artifacts={
                    "model": "constant.json",
                    "fit_timing": "fit_timing.json",
                },
                loader=_load_constant_checkpoint,
            )
            checkpoint_load_seconds += time.perf_counter() - checkpoint_started
            if valid is not None:
                fit_seconds = load_label_fit_timing(fit_timing_path)
            else:
                fit_seconds = 0.0
                atomic_write_json(
                    constant_path, {"constant": int(constant_labels[label_index])}
                )
                atomic_write_json(fit_timing_path, {"fit_seconds": fit_seconds})
                write_label_checkpoint(
                    attempt,
                    label_index=label_index,
                    label_name=label_name,
                    configuration=checkpoint_configuration,
                    observed_classes=observed_classes,
                    artifact_paths={
                        "model": constant_path,
                        "fit_timing": fit_timing_path,
                    },
                )
            label_fit_seconds.append(fit_seconds)
            continue

        if framework == "flaml":
            relative_artifact = "estimator.joblib"
            artifact_path = checkpoint_dir / relative_artifact
            loader = _load_flaml_checkpoint
        elif framework == "autogluon":
            relative_artifact = _marker_artifact(checkpoint_dir, "missing-predictor")
            artifact_path = checkpoint_dir / relative_artifact
            loader = _load_autogluon_checkpoint
        else:
            raise ValueError(f"Unsupported AutoML framework: {framework}")

        checkpoint_started = time.perf_counter()
        valid = validate_label_checkpoint(
            checkpoint_dir,
            unit_key=attempt.unit_key,
            label_index=label_index,
            label_name=label_name,
            configuration=checkpoint_configuration,
            observed_classes=observed_classes,
            expected_artifacts={
                "model": relative_artifact,
                "fit_timing": "fit_timing.json",
            },
            loader=loader,
        )
        checkpoint_load_seconds += time.perf_counter() - checkpoint_started
        if valid is not None:
            load_started = time.perf_counter()
            model = loader(artifact_path)
            fit_seconds = load_label_fit_timing(fit_timing_path)
            checkpoint_load_seconds += time.perf_counter() - load_started
        elif framework == "flaml":
            fit_started = time.perf_counter()
            model = _fit_flaml_label(
                X_train,
                y_train[:, label_index],
                budget_seconds=AUTOML_PER_LABEL_BUDGET_SECONDS,
                metric="f1",
                model_seed=model_seed,
                model_path=artifact_path,
            )
            fit_seconds = time.perf_counter() - fit_started
            atomic_write_json(fit_timing_path, {"fit_seconds": fit_seconds})
            write_label_checkpoint(
                attempt,
                label_index=label_index,
                label_name=label_name,
                configuration=checkpoint_configuration,
                observed_classes=observed_classes,
                artifact_paths={
                    "model": artifact_path,
                    "fit_timing": fit_timing_path,
                },
            )
        else:
            fresh_model_path = (
                attempt.path
                / "labels"
                / f"label-{label_index:03d}"
                / f"predictor-fresh-{label_index:03d}-{time.time_ns()}"
            )
            fit_started = time.perf_counter()
            model = _fit_autogluon_label(
                X_train,
                y_train[:, label_index],
                budget_seconds=AUTOML_PER_LABEL_BUDGET_SECONDS,
                metric="f1",
                model_seed=model_seed,
                model_path=fresh_model_path,
            )
            fit_seconds = time.perf_counter() - fit_started
            atomic_write_json(fit_timing_path, {"fit_seconds": fit_seconds})
            write_label_checkpoint(
                attempt,
                label_index=label_index,
                label_name=label_name,
                configuration=checkpoint_configuration,
                observed_classes=observed_classes,
                artifact_paths={
                    "model": fresh_model_path,
                    "fit_timing": fit_timing_path,
                },
            )
        label_fit_seconds.append(fit_seconds)
        fitted_label_models[label_index] = {
            "model": model,
            "observed_classes": observed_classes,
        }
    if len(label_fit_seconds) != EXPECTED_LABEL_COUNT:
        raise ValueError("Exactly 53 label fit timings are required")
    return (
        fitted_label_models,
        active_label_indices,
        constant_labels,
        label_fit_seconds,
        checkpoint_load_seconds,
    )
def predict_automl_label_scores(
    fitted_label_models,
    framework,
    X_eval,
    active_label_indices,
    constant_labels,
):
    active_scores = []
    for label_index in active_label_indices:
        fitted = fitted_label_models[label_index]
        active_scores.append(
            _binary_positive_probability(
                fitted["model"], framework, X_eval, fitted["observed_classes"]
            )
        )
    compact_scores = (
        np.column_stack(active_scores)
        if active_scores
        else np.empty((len(X_eval), 0), dtype=float)
    )
    scores = expand_active_scores(
        compact_scores,
        EXPECTED_LABEL_COUNT,
        active_label_indices,
        constant_labels,
    )
    if scores.shape != (len(X_eval), EXPECTED_LABEL_COUNT):
        raise ValueError(f"AutoML score expansion is not exactly 53 columns: {scores.shape}")
    return scores


def detect_available_hardware():
    hardware = {
        "cpu_count": os.cpu_count(),
        "cuda": False,
        "cuda_device_count": 0,
        "mps": False,
    }
    try:
        import torch

        hardware["cuda"] = bool(torch.cuda.is_available())
        hardware["cuda_device_count"] = int(torch.cuda.device_count())
        hardware["mps"] = bool(
            hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
        )
    except ImportError:
        pass
    return hardware


def detect_compute_device(available_hardware):
    if available_hardware["cuda"]:
        return "cuda"
    if available_hardware["mps"]:
        return "mps"
    return "cpu"
def validate_unit_artifacts(
    metrics, predictions, scores, timing, train_support, eval_support, model_config, unit_spec
):
    authoritative_labels = list(unit_spec["authoritative_labels"])
    eval_course_ids = list(unit_spec["eval_course_ids"])
    if len(authoritative_labels) != EXPECTED_LABEL_COUNT or len(authoritative_labels) != 53:
        raise ValueError("Unit must retain the authoritative 53-label order")
    if len(metrics) != len(DEFAULT_K_VALUES) or set(metrics["k"]) != set(DEFAULT_K_VALUES):
        raise ValueError("Metrics must contain exactly the configured fixed-53 K coordinates")
    expected_scores = {"courseId", *{f"score::{label}" for label in authoritative_labels}}
    if set(scores.columns) != expected_scores:
        raise ValueError("Score columns differ from the authoritative 53-label schema")
    if scores.shape != (len(eval_course_ids), EXPECTED_LABEL_COUNT + 1):
        raise ValueError("Score frame has an invalid shape")
    if scores["courseId"].astype(str).tolist() != eval_course_ids:
        raise ValueError("Score course IDs differ from the hashed evaluation order")
    if not np.isfinite(scores.drop(columns="courseId").to_numpy(float)).all():
        raise ValueError("Scores contain non-finite values")
    expected_prediction_coordinates = {
        (course_id, k) for course_id in eval_course_ids for k in DEFAULT_K_VALUES
    }
    actual_prediction_coordinates = {
        (str(row.courseId), int(row.k))
        for row in predictions[["courseId", "k"]].itertuples(index=False)
    }
    if actual_prediction_coordinates != expected_prediction_coordinates:
        raise ValueError("Prediction coordinates are incomplete or duplicated")
    if len(timing) != 1 or set(timing.columns) != {
        "train_seconds", "inference_seconds", "checkpoint_load_seconds",
        "total_seconds", "n_train", "n_eval"
    }:
        raise ValueError("Timing schema is invalid")
    expected_total_seconds = (
        float(timing.iloc[0]["train_seconds"])
        + float(timing.iloc[0]["inference_seconds"])
        + float(timing.iloc[0]["checkpoint_load_seconds"])
    )
    if not np.isclose(float(timing.iloc[0]["total_seconds"]), expected_total_seconds):
        raise ValueError("Timing total_seconds differs from measured components")
    if (
        len(train_support) != EXPECTED_LABEL_COUNT
        or train_support["label"].tolist() != authoritative_labels
    ):
        raise ValueError("Train support does not retain the authoritative order")
    if (
        len(eval_support) != EXPECTED_LABEL_COUNT
        or eval_support["label"].tolist() != authoritative_labels
    ):
        raise ValueError("Evaluation support does not retain the authoritative order")
    if model_config != unit_spec["model_config"]:
        raise ValueError("Model configuration differs from the hashed unit specification")
    expected_fields = {
        "experiment", "notebook", "model", "framework", "representation",
        "coordinates", "assignment_seed", "model_seed", "model_config",
        "authoritative_labels", "train_course_ids", "eval_course_ids",
        "source_csv_hashes", "notebook_code_fingerprint", "pipeline_utils_hash",
        "package_versions", "feature_identity", "protocol_version",
    }
    if set(unit_spec) != expected_fields:
        raise ValueError("Unit specification schema is invalid")
    return True


## Resumable unit lifecycle and strict aggregates

An attempt is selected only after all artifacts and the fixed label partition validate. Aggregates reject any missing, invalid, duplicated, or unexpected coordinate.


In [ ]:
def run_resumable_unit(
    output_dir,
    experiment,
    framework,
    representation,
    coordinates,
    assignment_seed,
    model_seed,
    train_indices,
    eval_indices,
    progress_counters,
):
    model_config = build_automl_model_config(framework, model_seed)
    model_config = {"seed": model_seed, **model_config}
    feature_identity = build_feature_identity(representation, model_seed)
    unit_spec = {
        "experiment": experiment,
        "notebook": "autoML.ipynb",
        "model": framework,
        "framework": framework,
        "representation": representation,
        "coordinates": coordinates,
        "assignment_seed": assignment_seed,
        "model_seed": model_seed,
        "model_config": model_config,
        "authoritative_labels": label_classes,
        "train_course_ids": course_ids[train_indices].tolist(),
        "eval_course_ids": course_ids[eval_indices].tolist(),
        "source_csv_hashes": source_csv_hashes,
        "notebook_code_fingerprint": notebook_fingerprint,
        "pipeline_utils_hash": pipeline_utils_hash,
        "package_versions": scientific_package_versions,
        "feature_identity": feature_identity,
        "protocol_version": PROTOCOL_VERSION,
    }
    unit_key = experiment_unit_key(unit_spec)
    effective_force_rerun = FORCE_RERUN or not RESUME_COMPLETED_UNITS
    attempt = prepare_experiment_attempt(
        output_dir, unit_spec, force_rerun=effective_force_rerun
    )
    if not attempt.should_run:
        progress_counters["skipped"] += 1
        report_progress_counters(progress_counters)
        return attempt.unit_dir
    try:
        X_train, X_eval, observed_feature_identity = prepare_unit_features(
            representation, train_indices, eval_indices, model_seed=model_seed
        )
        if observed_feature_identity != feature_identity:
            raise ValueError("Resolved feature identity differs from the hashed unit specification")
        run_bert_identity = {
            "requested_model": BERT_MODEL_ID,
            "resolved_model_revision": bert_cache_metadata["resolved_model_revision"],
            "requested_tokenizer": BERT_MODEL_ID,
            "resolved_tokenizer_revision": bert_cache_metadata["resolved_tokenizer_revision"],
        }
        run_embedding_cache_identity = {
            "cache_key": bert_cache_metadata["cache_key"],
            "source_sha256": bert_cache_metadata["source_sha256"],
            "ordered_course_ids_sha256": bert_cache_metadata["ordered_course_ids_sha256"],
            "embedding_sha256": bert_cache_metadata["embedding_sha256"],
            "index_column": "courseId",
        }
        available_hardware = actual_device = detect_available_hardware()
        actual_device = "cpu"
        provenance = collect_run_provenance(
            list(scientific_package_versions),
            device=actual_device,
            available_hardware=available_hardware,
            bert_identity=run_bert_identity,
            embedding_cache_identity=run_embedding_cache_identity,
        )
        provenance["training_order_seed"] = model_config["training_order_seed"]
        run_manifest = build_run_manifest(unit_key, unit_spec, provenance)
        validate_run_manifest(run_manifest)
        active_label_indices, constant_labels = validate_label_partition_manifest(
            y_all[train_indices]
        )
        unit_manifest = build_unit_manifest(
            unit_spec, active_label_indices, constant_labels
        )
        training_started = time.perf_counter()
        (
            fitted_label_models,
            trained_active_indices,
            trained_constants,
            label_fit_seconds,
            checkpoint_load_seconds,
        ) = run_automl_label_searches(
            framework,
            X_train,
            y_all[train_indices],
            label_classes,
            attempt,
            model_config=model_config,
            model_seed=model_seed,
        )
        train_seconds = sum(label_fit_seconds)
        if trained_active_indices != active_label_indices or trained_constants != constant_labels:
            raise ValueError("Training label partition differs from the unit manifest")
        inference_started = time.perf_counter()
        scores = predict_automl_label_scores(
            fitted_label_models,
            framework,
            X_eval,
            trained_active_indices,
            trained_constants,
        )
        inference_seconds = time.perf_counter() - inference_started
        metrics, predictions = compute_multilabel_metrics(
            y_all[eval_indices],
            scores,
            label_classes,
            method=f"{framework}_{representation}",
            k_values=DEFAULT_K_VALUES,
        )
        metrics["fold"] = coordinates.get("fold")
        metrics["training_size"] = coordinates.get("training_size")
        metrics["repetition"] = coordinates.get("repetition")
        predictions["courseId"] = np.tile(
            course_ids[eval_indices], len(DEFAULT_K_VALUES)
        )
        score_frame = pd.DataFrame(
            scores, columns=[f"score::{label}" for label in label_classes]
        )
        score_frame.insert(0, "courseId", course_ids[eval_indices])
        total_seconds = train_seconds + inference_seconds + checkpoint_load_seconds
        timing = pd.DataFrame([{
            "train_seconds": train_seconds,
            "inference_seconds": inference_seconds,
            "checkpoint_load_seconds": checkpoint_load_seconds,
            "total_seconds": total_seconds,
            "n_train": len(train_indices),
            "n_eval": len(eval_indices),
        }])
        train_support = label_support_summary(
            y_all[train_indices], label_classes
        )
        eval_support = label_support_summary(
            y_all[eval_indices], label_classes
        )
        validate_unit_artifacts(
            metrics,
            predictions,
            score_frame,
            timing,
            train_support,
            eval_support,
            model_config,
            unit_spec,
        )
        artifact_frames = {
            "metrics": metrics,
            "predictions": predictions,
            "scores": score_frame,
            "timing": timing,
            "train_support": train_support,
            "eval_support": eval_support,
        }
        artifact_paths = {}
        artifact_counts = {}
        for artifact_name, frame in artifact_frames.items():
            artifact_path = attempt.path / f"{artifact_name}.csv"
            atomic_write_csv(artifact_path, artifact_frames[artifact_name])
            artifact_paths[artifact_name] = artifact_path
            artifact_counts[artifact_name] = len(frame)
        run_manifest_path = attempt.path / "run_manifest.json"
        unit_manifest_path = attempt.path / "unit_manifest.json"
        model_config_path = attempt.path / "model_config.json"
        unit_spec_path = attempt.path / "unit_spec.json"
        atomic_write_json(run_manifest_path, run_manifest)
        atomic_write_json(unit_manifest_path, unit_manifest)
        atomic_write_json(model_config_path, model_config)
        artifact_paths.update({
            "run_manifest": run_manifest_path,
            "unit_manifest": unit_manifest_path,
            "model_config": model_config_path,
            "unit_spec": unit_spec_path,
        })
        artifact_counts.update({
            "run_manifest": len(run_manifest),
            "unit_manifest": len(unit_manifest),
            "model_config": len(model_config),
            "unit_spec": len(unit_spec),
        })
        complete_experiment_attempt(
            attempt,
            artifact_paths=artifact_paths,
            artifact_counts=artifact_counts,
            provenance=provenance,
            active_label_indices=active_label_indices,
            constant_labels=constant_labels,
        )
        progress_counters["completed"] += 1
        report_progress_counters(progress_counters)
        return attempt.unit_dir
    except Exception as error:
        write_attempt_failure(attempt, error)
        progress_counters["failed"] += 1
        report_progress_counters(progress_counters)
        raise


def require_complete_selected_attempts(unit_dirs):
    selected_attempts = []
    missing = []
    invalid = []
    for unit_dir in unit_dirs:
        selected = selected_attempt_path(unit_dir)
        if selected is None:
            missing.append(str(unit_dir))
            continue
        validated = validate_completed_attempt(selected, expected_unit_key=unit_dir.name)
        if validated is None:
            invalid.append(str(unit_dir))
            raise ValueError(f"Invalid selected attempt: {invalid}")
        selected_attempts.append(selected)
    if missing or invalid:
        raise ValueError(
            f"Missing selected attempts: {missing}; invalid selected attempts: {invalid}"
        )
    return selected_attempts


def publish_current_curve_selection(
    output_dir,
    selected_attempts,
    *,
    expected_unit_keys,
    package_versions,
    family,
    framework,
):
    expected_unit_keys = set(expected_unit_keys)
    selected_unit_keys = {attempt.parent.name for attempt in selected_attempts}
    if selected_unit_keys != expected_unit_keys:
        raise ValueError(
            f"Curve selection mismatch; missing={expected_unit_keys - selected_unit_keys}; "
            f"extra={selected_unit_keys - expected_unit_keys}"
        )
    artifact_hashes = {}
    selected_attempt_keys = {}
    for selected_attempt in selected_attempts:
        completion = validate_completed_attempt(
            selected_attempt, expected_unit_key=selected_attempt.parent.name
        )
        if completion is None:
            raise ValueError(f"Invalid selected curve attempt: {selected_attempt}")
        unit_key = selected_attempt.parent.name
        selected_attempt_keys[unit_key] = selected_attempt.name
        artifact_hashes[unit_key] = {
            "completed.json": file_sha256(selected_attempt / "completed.json"),
            **{
                artifact_name: artifact["sha256"]
                for artifact_name, artifact in completion["artifacts"].items()
            },
        }
    manifest = {
        "schema_version": "curve-selection-v1",
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "family": family,
        "framework": framework,
        "protocol_key": protocol["manifest"]["protocol_key"],
        "protocol_version": PROTOCOL_VERSION,
        "authoritative_labels": list(label_classes),
        "source_csv_hashes": dict(source_csv_hashes),
        "train_course_ids": train_data["courseId"].astype(str).tolist(),
        "eval_course_ids": test_data["courseId"].astype(str).tolist(),
        "notebook_code_fingerprint": notebook_fingerprint,
        "pipeline_utils_hash": pipeline_utils_hash,
        "package_versions": dict(package_versions),
        "expected_unit_keys": sorted(expected_unit_keys),
        "selected_unit_keys": sorted(selected_unit_keys),
        "artifact_hashes": artifact_hashes,
        "selected_attempt_keys": selected_attempt_keys,
    }
    selection_key = canonical_payload_hash(manifest)
    selection_dir = output_dir / "selection_manifests"
    selection_dir.mkdir(parents=True, exist_ok=True)
    selection_manifest_path = selection_dir / f"{selection_key}.json"
    if selection_manifest_path.exists():
        existing_manifest = json.loads(
            selection_manifest_path.read_text(encoding="utf-8")
        )
        if existing_manifest != manifest:
            raise ValueError("Curve selection manifest hash collision")
    else:
        atomic_write_json(selection_manifest_path, manifest)
    atomic_write_json(output_dir / "current_curve_selection.json", {
        "schema_version": "curve-selection-pointer-v1",
        "selection_key": selection_key,
        "manifest_path": str(selection_manifest_path),
        "manifest_sha256": file_sha256(selection_manifest_path),
    })
    return manifest


def rebuild_cv_aggregate(output_dir, unit_dirs):
    selected_attempts = require_complete_selected_attempts(unit_dirs)
    selected_attempts = list(selected_attempts)
    metric_frames = []
    score_frames = []
    assignment_frames = []
    for selected in selected_attempts:
        unit_spec = read_selected_artifact(selected, "unit_spec", kind="json")
        metrics = read_selected_artifact(selected, "metrics", kind="csv")
        scores = read_selected_artifact(selected, "scores", kind="csv")
        metrics["framework"] = unit_spec["framework"]
        metrics["representation"] = unit_spec["representation"]
        metrics["fold"] = unit_spec["coordinates"]["fold"]
        scores["framework"] = unit_spec["framework"]
        scores["representation"] = unit_spec["representation"]
        scores["fold"] = unit_spec["coordinates"]["fold"]
        metric_frames.append(metrics)
        score_frames.append(scores)
        assignment_frames.append(pd.DataFrame({
            "courseId": unit_spec["eval_course_ids"],
            "fold": unit_spec["coordinates"]["fold"],
            "framework": unit_spec["framework"],
            "representation": unit_spec["representation"],
        }))
    fold_metrics = pd.concat(metric_frames, ignore_index=True)
    duplicate_cv_coordinates = fold_metrics.duplicated(
        subset=["framework", "representation", "fold", "k"], keep=False
    )
    if duplicate_cv_coordinates.any():
        raise ValueError("CV metrics contain duplicate full coordinate rows")
    expected_cv_coordinates = {
        (framework, representation, fold, k)
        for framework, representation in AUTOML_METHOD_MATRIX
        for fold in range(CV_N_SPLITS)
        for k in DEFAULT_K_VALUES
    }
    actual_cv_coordinates = {
        (row.framework, row.representation, int(row.fold), int(row.k))
        for row in fold_metrics[["framework", "representation", "fold", "k"]].itertuples(index=False)
    }
    if len(fold_metrics) != len(expected_cv_coordinates):
        raise ValueError("CV metric row count differs from exact expected coordinates")
    if actual_cv_coordinates != expected_cv_coordinates:
        raise ValueError(
            f"CV coordinate mismatch; missing={expected_cv_coordinates - actual_cv_coordinates}; "
            f"extra={actual_cv_coordinates - expected_cv_coordinates}"
        )
    numeric = [
        "micro_f1", "macro_f1", "hamming_loss", "subset_accuracy",
        "precision_at_k", "recall_at_k", "partial_hit_at_k",
    ]
    grouping = ["framework", "representation", "k"]
    cv_mean = fold_metrics.groupby(grouping)[numeric].mean().add_suffix("_mean")
    cv_std = fold_metrics.groupby(grouping)[numeric].std(ddof=1).add_suffix("_std")
    summary = cv_mean.join(cv_std).reset_index()
    fold_assignments = pd.concat(assignment_frames, ignore_index=True)
    atomic_write_csv(output_dir / "fold_metrics.csv", fold_metrics)
    atomic_write_csv(output_dir / "mean_std_metrics.csv", summary)
    atomic_write_csv(output_dir / "fold_assignments.csv", fold_assignments)
    score_table = pd.concat(score_frames, ignore_index=True)
    score_columns = [f"score::{label}" for label in label_classes]
    pooled_frames = []
    for (framework, representation), group in score_table.groupby(["framework", "representation"]):
        if set(group["courseId"].astype(str)) != set(course_ids):
            raise ValueError("Pooled OOF scores do not cover the exact 338-course universe")
        ordered = group.set_index("courseId").loc[course_ids]
        pooled_scores = ordered[score_columns].to_numpy(float)
        pooled_oof, _ = compute_multilabel_metrics(
            y_all,
            pooled_scores,
            label_classes,
            method=f"{framework}_{representation}_pooled_oof",
            k_values=DEFAULT_K_VALUES,
        )
        pooled_oof["framework"] = framework
        pooled_oof["representation"] = representation
        pooled_oof["aggregate"] = "pooled_oof"
        pooled_frames.append(pooled_oof)
    atomic_write_csv(output_dir / "pooled_oof_metrics.csv", pd.concat(pooled_frames, ignore_index=True))
    atomic_write_csv(output_dir / "oof_scores.csv", score_table)
    return summary


def rebuild_curve_aggregate(output_dir, unit_dirs):
    selected_attempts = require_complete_selected_attempts(unit_dirs)
    selected_attempts = list(selected_attempts)
    frames = []
    for selected in selected_attempts:
        unit_spec = read_selected_artifact(selected, "unit_spec", kind="json")
        frame = read_selected_artifact(selected, "metrics", kind="csv")
        frame["framework"] = unit_spec["framework"]
        frame["representation"] = unit_spec["representation"]
        frame["training_size"] = unit_spec["coordinates"]["training_size"]
        frame["repetition"] = unit_spec["coordinates"]["repetition"]
        frames.append(frame)
    curve_metrics = pd.concat(frames, ignore_index=True)
    duplicate_curve_coordinates = curve_metrics.duplicated(
        subset=["framework", "representation", "training_size", "repetition", "k"],
        keep=False,
    )
    if duplicate_curve_coordinates.any():
        raise ValueError("Curve metrics contain duplicate full coordinate rows")
    expected_curve_coordinates = {
        (framework, representation, size, repetition, k)
        for framework, representation in AUTOML_METHOD_MATRIX
        for size in TRAIN_SIZES
        for repetition in range(TRAIN_SIZE_REPETITIONS)
        for k in DEFAULT_K_VALUES
    }
    actual_curve_coordinates = {
        (
            row.framework,
            row.representation,
            int(row.training_size),
            int(row.repetition),
            int(row.k),
        )
        for row in curve_metrics[
            ["framework", "representation", "training_size", "repetition", "k"]
        ].itertuples(index=False)
    }
    if len(curve_metrics) != len(expected_curve_coordinates):
        raise ValueError("Curve metric row count differs from exact expected coordinates")
    if actual_curve_coordinates != expected_curve_coordinates:
        raise ValueError(
            f"Curve coordinate mismatch; missing={expected_curve_coordinates - actual_curve_coordinates}; "
            f"extra={actual_curve_coordinates - expected_curve_coordinates}"
        )
    if curve_metrics["repetition"].nunique() != TRAIN_SIZE_REPETITIONS:
        raise ValueError("Training-size curve is incomplete")
    numeric = [
        "micro_f1", "macro_f1", "hamming_loss", "subset_accuracy",
        "precision_at_k", "recall_at_k", "partial_hit_at_k",
    ]
    grouping = ["framework", "representation", "training_size", "k"]
    curve_mean = curve_metrics.groupby(grouping)[numeric].mean().add_suffix("_mean")
    curve_std = curve_metrics.groupby(grouping)[numeric].std(ddof=1).add_suffix("_std")
    curve_lower = curve_metrics.groupby(grouping)[numeric].quantile(0.025).add_suffix("_lower")
    curve_upper = curve_metrics.groupby(grouping)[numeric].quantile(0.975).add_suffix("_upper")
    summary = curve_mean.join(curve_std).join(curve_lower).join(curve_upper).reset_index()
    atomic_write_csv(output_dir / "training_size_repetitions.csv", curve_metrics)
    atomic_write_csv(output_dir / "training_size_summary.csv", summary)
    publish_current_curve_selection(
        output_dir,
        selected_attempts,
        expected_unit_keys={Path(unit_dir).name for unit_dir in unit_dirs},
        package_versions=scientific_package_versions,
        family="automl",
        framework="flaml+autogluon",
    )
    return summary


def validate_full_training_repetition_invariants(records):
    full_records = [record for record in records if record["training_size"] == 270]
    membership_signatures = {record["membership_signature"] for record in full_records}
    unit_keys = {record["unit_key"] for record in full_records}
    subset_seeds = {record["subset_seed"] for record in full_records}
    model_seeds = {record["model_seed"] for record in full_records}
    repetitions = {record["repetition"] for record in full_records}
    if len(membership_signatures) != 1:
        raise ValueError("Full-training membership must be identical across repetitions")
    if (
        len(unit_keys) != 3
        or len(subset_seeds) != 3
        or len(model_seeds) != 3
        or len(repetitions) != 3
    ):
        raise ValueError("Full-training unit keys, seeds, and repetitions must remain distinct")
    return full_records


## Validated data and shared assignment bundle


In [ ]:
EXPERIMENT_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
required_paths = [Path("processed.csv"), Path("train.csv"), Path("test.csv")]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(f"Preflight missing files: {missing_paths}")

processed_data = load_processed_csv("processed.csv")
train_data = load_split_csv("train.csv")
test_data = load_split_csv("test.csv")
required_columns = {"courseId", "combinedText", "comp_name_filtered"}
column_errors = {
    name: sorted(required_columns - set(frame.columns))
    for name, frame in (
        ("processed", processed_data),
        ("train", train_data),
        ("test", test_data),
    )
    if required_columns - set(frame.columns)
}
if column_errors:
    raise ValueError(f"Preflight missing columns: {column_errors}")
expected_counts = {"processed": 338, "train": 270, "test": 68}
observed_counts = {
    "processed": len(processed_data),
    "train": len(train_data),
    "test": len(test_data),
}
if observed_counts != expected_counts:
    raise ValueError(f"Dataset counts differ: {observed_counts}")
validate_dataset_identity(processed_data, train_data, test_data)
label_classes = authoritative_label_classes(
    processed_data, expected_n_labels=EXPECTED_LABEL_COUNT
)
course_ids = processed_data["courseId"].astype(str).to_numpy()
y_all = fixed_label_matrix(
    processed_data["comp_name_filtered"], label_classes
)
source_csv_hashes = {
    name: file_sha256(name) for name in ("processed.csv", "train.csv", "test.csv")
}
protocol = ensure_experiment_protocol(
    EXPERIMENT_OUTPUT_ROOT / "protocol",
    processed_df=processed_data,
    train_df=train_data,
    test_df=test_data,
    y=y_all,
    authoritative_labels=label_classes,
    source_csv_hashes=source_csv_hashes,
    assignment_seed=PROTOCOL_ASSIGNMENT_SEED,
    model_seed=EXPERIMENT_SEED,
    n_splits=CV_N_SPLITS,
    training_sizes=TRAIN_SIZES,
    repetitions=TRAIN_SIZE_REPETITIONS,
    expected_counts={
        "courses": 338,
        "train_courses": 270,
        "test_courses": 68,
        "labels": 53,
    },
    protocol_version=PROTOCOL_VERSION,
)
fold_assignments = protocol["fold_assignments"]
training_subsets = protocol["training_subsets"]

BERT_MODEL_ID = "neuralmind/bert-base-portuguese-cased"
BERT_MAX_LENGTH = 128
BERT_BATCH_SIZE = 32
BERT_HIDDEN_SIZE = 768
bert_embeddings, bert_cache_metadata = load_or_build_indexed_bert_cache(
    course_ids,
    processed_data["combinedText"].astype(str).tolist(),
)
bert_identity = {
    "requested_model": BERT_MODEL_ID,
    "resolved_model_revision": bert_cache_metadata["resolved_model_revision"],
    "requested_tokenizer": BERT_MODEL_ID,
    "resolved_tokenizer_revision": bert_cache_metadata["resolved_tokenizer_revision"],
}
embedding_cache_identity = {
    "cache_key": bert_cache_metadata["cache_key"],
    "source_sha256": bert_cache_metadata["source_sha256"],
    "ordered_course_ids_sha256": bert_cache_metadata["ordered_course_ids_sha256"],
    "embedding_sha256": bert_cache_metadata["embedding_sha256"],
    "index_column": "courseId",
}
scientific_package_versions = {}
for package_name in [
    "numpy", "pandas", "scikit-learn", "flaml", "autogluon.tabular",
    "gensim", "torch", "transformers",
]:
    try:
        scientific_package_versions[package_name] = importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        scientific_package_versions[package_name] = "not-installed"
notebook_fingerprint = notebook_code_fingerprint("autoML.ipynb")
pipeline_utils_hash = file_sha256("pipeline_utils.py")


## Full CV and training-size executions


In [ ]:
cv_dir = EXPERIMENT_OUTPUT_ROOT / "stratified_cv" / "automl"
curve_dir = EXPERIMENT_OUTPUT_ROOT / "training_size_curve" / "automl"
cv_dir.mkdir(parents=True, exist_ok=True)
curve_dir.mkdir(parents=True, exist_ok=True)
cv_progress_counters = {
    "total": len(AUTOML_METHOD_MATRIX) * CV_N_SPLITS,
    "completed": 0,
    "skipped": 0,
    "failed": 0,
    "remaining": 0,
}
curve_progress_counters = {
    "total": len(AUTOML_METHOD_MATRIX) * len(TRAIN_SIZES) * TRAIN_SIZE_REPETITIONS,
    "completed": 0,
    "skipped": 0,
    "failed": 0,
    "remaining": 0,
}
fixed_test_ids = set(test_data["courseId"].astype(str))
fixed_test_idx = np.array([
    index for index, course_id in enumerate(course_ids)
    if course_id in fixed_test_ids
])
full_train_course_ids = train_data["courseId"].astype(str).tolist()


def _run_framework_cv(framework, method_matrix):
    unit_dirs = []
    assignment_by_id = fold_assignments.set_index("courseId")["fold"]
    representations = [
        representation
        for candidate, representation in method_matrix
        if candidate == framework
    ]
    for fold in range(CV_N_SPLITS):
        cv_model_seed = EXPERIMENT_SEED + fold
        valid_ids = set(assignment_by_id.index[assignment_by_id == fold].astype(str))
        eval_indices = np.array([
            index for index, course_id in enumerate(course_ids)
            if course_id in valid_ids
        ])
        eval_set = set(eval_indices)
        train_indices = np.array([
            index for index in range(len(course_ids)) if index not in eval_set
        ])
        for representation in representations:
            unit_dirs.append(run_resumable_unit(
                cv_dir,
                experiment="stratified-cv",
                framework=framework,
                representation=representation,
                coordinates={"fold": fold, "training_size": None, "repetition": None},
                assignment_seed=PROTOCOL_ASSIGNMENT_SEED,
                model_seed=EXPERIMENT_SEED + fold,
                train_indices=train_indices,
                eval_indices=eval_indices,
                progress_counters=cv_progress_counters,
            ))
    return unit_dirs


def _run_framework_curve(framework, method_matrix):
    unit_dirs = []
    full_training_records = []
    representations = [
        representation
        for candidate, representation in method_matrix
        if candidate == framework
    ]
    for repetition in range(TRAIN_SIZE_REPETITIONS):
        curve_subset_seed = EXPERIMENT_SEED + repetition
        curve_model_seed = EXPERIMENT_SEED + repetition
        repetition_rows = training_subsets[
            training_subsets["repetition"] == repetition
        ]
        for size in TRAIN_SIZES:
            selected_ids = repetition_rows.loc[
                repetition_rows["training_size"] == size, "courseId"
            ].astype(str).tolist()
            if size == 270 and selected_ids != full_train_course_ids:
                if set(selected_ids) != set(full_train_course_ids):
                    raise ValueError("The size-270 subset must equal the fixed training set")
                selected_ids = full_train_course_ids
            selected_set = set(selected_ids)
            train_indices = np.array([
                index for index, course_id in enumerate(course_ids)
                if course_id in selected_set
            ])
            for representation in representations:
                unit_dir = run_resumable_unit(
                    curve_dir,
                    experiment="training-curve",
                    framework=framework,
                    representation=representation,
                    coordinates={
                        "fold": None,
                        "training_size": size,
                        "repetition": repetition,
                    },
                    assignment_seed=EXPERIMENT_SEED + repetition,
                    model_seed=EXPERIMENT_SEED + repetition,
                    train_indices=train_indices,
                    eval_indices=fixed_test_idx,
                    progress_counters=curve_progress_counters,
                )
                unit_dirs.append(unit_dir)
                if size == 270:
                    full_training_records.append({
                        "framework": framework,
                        "representation": representation,
                        "training_size": size,
                        "membership_signature": tuple(sorted(selected_ids)),
                        "unit_key": unit_dir.name,
                        "subset_seed": curve_subset_seed,
                        "model_seed": curve_model_seed,
                        "repetition": repetition,
                    })
    for representation in representations:
        validate_full_training_repetition_invariants([
            record for record in full_training_records
            if record["representation"] == representation
        ])
    return unit_dirs


def run_flaml(phase, method_matrix):
    if phase == "cv":
        return _run_framework_cv("flaml", method_matrix)
    if phase == "curve":
        return _run_framework_curve("flaml", method_matrix)
    raise ValueError(f"Unknown execution phase: {phase}")


def run_autogluon(phase, method_matrix):
    if phase == "cv":
        return _run_framework_cv("autogluon", method_matrix)
    if phase == "curve":
        return _run_framework_curve("autogluon", method_matrix)
    raise ValueError(f"Unknown execution phase: {phase}")


def collect_cv_results(framework_results):
    unit_dirs = [path for paths in framework_results.values() for path in paths]
    return rebuild_cv_aggregate(cv_dir, unit_dirs)


def collect_curve_results(framework_results):
    unit_dirs = [path for paths in framework_results.values() for path in paths]
    return rebuild_curve_aggregate(curve_dir, unit_dirs)


phase_outputs = {}
for phase_name, method_matrix in [
    ("cv", AUTOML_METHOD_MATRIX),
    ("curve", AUTOML_METHOD_MATRIX),
]:
    framework_results = {
        "flaml": run_flaml(phase_name, method_matrix),
        "autogluon": run_autogluon(phase_name, method_matrix),
    }
    if phase_name == "cv":
        phase_outputs[phase_name] = collect_cv_results(framework_results)
    else:
        phase_outputs[phase_name] = collect_curve_results(framework_results)
